# **Multi-Brand Marketing Campaign Performance Analysis**

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib

In [ ]:
final_df = pd.read_csv(r"D:\PROJECTS\Anna_Project_3\Multi-Brand_Marketing_Campaign_Performance_Analysis\CSV\final_df.csv",index_col=0)

# **Exploratory Data Analysis (EDA)**

### **Analyze campaign performance across brands**

In [ ]:
analysis_2 = (
    final_df
    .groupby('Campaign_Type')['ROI']
    .mean()
    .reset_index()
    .sort_values(by='ROI', ascending=False)
)

fig = px.bar(
    analysis_2,
    x='Campaign_Type',
    y='ROI',
    color='ROI',
    title='Average ROI Across Brands'
)

fig.show()

### **Identify top-performing and low-performing campaigns**

In [ ]:
top_campaigns = (
    final_df
    .groupby('Campaign_Type')['ROI']
    .mean()
    .nlargest(5)
    .reset_index()
)

top_campaigns

In [ ]:
low_campaigns = (
    final_df
    .groupby('Campaign_Type')['ROI']
    .mean()
    .nsmallest(5)
    .reset_index()
)

low_campaigns

### **Explore relationships between spend, clicks, revenue, and ROI**

In [ ]:
final_df.columns

In [ ]:
final_df[['Acquisition_Cost','Clicks','Revenue','ROI']].corr()

In [ ]:
corr_matrix = final_df[['Acquisition_Cost', 'Clicks', 'Revenue', 'ROI']].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    title='Correlation Between Spend, Clicks, Revenue, and ROI'
)

fig.show()

### **Analyze channel-wise effectiveness**

In [ ]:
analysis_channel = (
    final_df
    .groupby('Channel_Used')
    .agg({
        'Revenue': 'sum',
        'ROI': 'mean',
        'Clicks': 'sum',
        'Conversions': 'sum',
        'Leads': 'sum',
        'Impressions': 'sum'
    })
    .reset_index()
    .sort_values(by='ROI', ascending=False)
)

analysis_channel

In [ ]:
channel_df = final_df.copy()

channel_df['Channel_Used'] = channel_df['Channel_Used'].str.split(', ')

channel_df = channel_df.explode('Channel_Used')

analysis_channel = (
    channel_df
    .groupby('Channel_Used')
    .agg({
        'Revenue': 'sum',
        'ROI': 'mean',
        'Clicks': 'sum',
        'Conversions': 'sum',
        'Leads': 'sum',
        'Impressions': 'sum'
    })
    .reset_index()
    .sort_values(by='ROI', ascending=False)
)

analysis_channel

#### **Encoding**

In [ ]:
final_df.select_dtypes("object").columns

In [ ]:
label_columns = ['Campaign_Type', 'Target_Audience', 'Language', 'Customer_Segment']
label_encoders = {}
 
for col in label_columns:
    le = LabelEncoder()
    final_df[col] = le.fit_transform(final_df[col])
    label_encoders[col] = le          # keep this column's own fitted encoder
 
joblib.dump(label_encoders, "label_encoders.pkl")
print("Label encoders saved successfully!")

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

channel_encoded = mlb.fit_transform(
    final_df["Channel_Used"].str.split(", ")
)

channel_df = pd.DataFrame(
    channel_encoded,
    columns=mlb.classes_,
    index=final_df.index
)

final_df = pd.concat([final_df.drop(columns=["Channel_Used"]), channel_df], axis=1)

In [ ]:
final_df['ROI_Flag'].value_counts()

In [ ]:
final_df['ROI_Flag'] = final_df['ROI_Flag'].map({'Profit':0,'Loss':1})

# **Model Building**

## **Regression Model**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import LinearSVR
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error,root_mean_squared_error

In [ ]:
final_df.columns

In [ ]:
X_r = final_df.drop(columns=['Campaign_ID','ROI','ROI_Flag','Revenue','Language','Campaign_Type','Date'])
y_r = final_df['Revenue']

In [ ]:
X_train_r,X_test_r,y_train_r,y_test_r = train_test_split(X_r,y_r,test_size=0.20,random_state=42)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_r)
X_test = scaler.transform(X_test_r)

In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    metrics = {
        "Model": name,
        "r2 Score": r2_score(y_test, preds),
        "MAE": mean_absolute_error(y_test, preds),
        "MSE": mean_squared_error(y_test, preds),
        "RMSE": root_mean_squared_error(y_test, preds)
    }

    print(f"--- {name} ---")
    print(f"r2 Score : {metrics['r2 Score']:.4f}")
    print(f"MAE: {metrics['MAE']:.4f}")
    print(f"MSE: {metrics['MSE']:.4f}")
    print(f"RMSE: {metrics['RMSE']:.4f}")

    print()

    return model, preds, metrics

In [ ]:
final_df.info()

In [ ]:
models = {
        "Linear Regression": LinearRegression(),
        "KNN": KNeighborsRegressor(n_neighbors=3),
        "Decision Tree": DecisionTreeRegressor(random_state=42),
        "Random Forest": RandomForestRegressor(random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
        "XGBoost": XGBRegressor(
            objective="reg:squarederror",
            random_state=42
        )
    }

results = []
predictions = {}
fitted_models = {}

for name, model in models.items():
    fitted_model, preds, metrics = evaluate_model(
        name, model, X_train, y_train_r, X_test, y_test_r
    )
    results.append(metrics)
    predictions[name] = preds
    fitted_models[name] = fitted_model

In [ ]:
final_df

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

best_model_name = results_df.loc[
    results_df["r2 Score"].idxmax(),
    "Model"
]

best_model = fitted_models[best_model_name]

print("Best Model:", best_model_name)
print("Best r2 Score:",
      results_df["r2 Score"].max())

In [ ]:
import joblib

joblib.dump(best_model, "best_Regression_model.pkl")
joblib.dump(scaler, "Regression_scaler.pkl")

print("Model saved successfully!")


In [ ]:
model_columns = X_r.columns.tolist()
joblib.dump(model_columns, "model_columns_regression.pkl")

## **Classification Model**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [ ]:
X_c = final_df.drop(
    columns=['Campaign_ID', 'ROI', 'ROI_Flag',
             'Language', 'Campaign_Type', 'Date']
)
y_c=final_df['ROI_Flag']

In [ ]:
final_df.columns

In [ ]:
X_train_c,X_test_c,y_train_c,y_test_c = train_test_split(X_c,y_c,test_size=0.20,random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_c = scaler.fit_transform(X_train_c)
X_test_c = scaler.transform(X_test_c)

In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1 Score": f1_score(y_test, preds, zero_division=0),
        "Confusion Matrix": confusion_matrix(y_test, preds)
    }

    print(f"--- {name} ---")
    print(f"Accuracy : {metrics['Accuracy']:.4f}")
    print(f"Precision: {metrics['Precision']:.4f}")
    print(f"Recall   : {metrics['Recall']:.4f}")
    print(f"F1 Score : {metrics['F1 Score']:.4f}")
    print("Confusion Matrix:")
    print(metrics["Confusion Matrix"])
    print()

    return model, preds, metrics

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ),

    "KNN": KNeighborsClassifier(n_neighbors=3),

    "Decision Tree": DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        class_weight='balanced',
        random_state=42
    ),

    "SVM": SVC(
        class_weight='balanced'
    ),

    "Naive Bayes": GaussianNB(),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        eval_metric='logloss',
        random_state=42
    )
}

results = []
predictions = {}
fitted_models = {}

for name, model in models.items():
    fitted_model, preds, metrics = evaluate_model(
        name, model, X_train_c, y_train_c, X_test_c, y_test_c
    )
    results.append(metrics)
    predictions[name] = preds
    fitted_models[name] = fitted_model

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

best_model_name = results_df.loc[
    results_df["F1 Score"].idxmax(),
    "Model"
]

best_model = fitted_models[best_model_name]

print("Best Model:", best_model_name)
print("Best F1 Score:",
      results_df["F1 Score"].max())

In [ ]:
import joblib

joblib.dump(best_model, "best_classification_model.pkl")
joblib.dump(scaler, "classification_scaler.pkl")


print("Model saved successfully!")

In [ ]:
model_columns = X_c.columns.tolist()
joblib.dump(model_columns, "model_columns_class.pkl")